In [ ]:
from src.post_processing import PathWrangler
import polars as pl
from pathlib import Path
from collections import defaultdict

In [ ]:
study = "/home/stef/quest_data/bottle/data/processed/lin_test"
known = "/home/stef/bottle/artifacts/known"
out_dir = "/home/stef/bottle/artifacts/coa_mutase_paths"

pw = PathWrangler(Path(study), Path(known))

In [ ]:
tables = pw.get_paths()

In [ ]:
paths = tables['paths']
krs = tables['known_reactions']
prs = tables['predicted_reactions']

pr2krs = dict(zip(prs['id'], prs['analogue_ids'].to_list()))
krs2enzymes = dict(zip(krs['id'], krs['enzymes'].to_list()))
prs2enzymes = defaultdict(list)
for p, ks in pr2krs.items():
    for k in ks:
        prs2enzymes[p] += krs2enzymes[k]

prs2smarts = dict(zip(prs['id'], prs['smarts'].to_list()))

paths = paths.select(
    pl.col("id"),
    pl.col("rxn_id"),
    (pl.col("generation") + 1).alias("step"),
    pl.col("starters").map_elements(lambda x : ";".join(x), return_dtype=pl.String),
    pl.col("targets").map_elements(lambda x : ";".join(x), return_dtype=pl.String),
    pl.col("rxn_id").replace_strict(prs2enzymes, default=[]).map_elements(lambda x : ";".join(x), return_dtype=pl.String).alias("enzymes"),
    pl.col("rxn_id").replace_strict(prs2smarts, default=None).alias("smarts")
)
paths

In [ ]:
enzymes = tables['enzymes']


enzymes.with_columns(
    pl.col("id").map_elements(lambda x : f"https://www.uniprot.org/uniprotkb/{x}/entry", return_dtype=pl.String).alias("link"),
)

In [ ]:
paths.write_csv(
    Path(out_dir) / "250912_3hpa_paths.csv",
    separator=','
)

enzymes.write_csv(
    Path(out_dir) / "250912_3hpa_enzymes.csv",
    separator=','
)

In [ ]:
from ergochemics.draw import draw_reaction
from IPython.display import SVG, display

In [ ]:
paths.filter(
    pl.col("id").str.starts_with("0a24a1c")
)

In [ ]:
for row in paths.filter(pl.col("id").str.starts_with("0a24a1c")).iter_rows(named=True):
    smarts = row['smarts']
    print(row['rxn_id'])
    display(SVG(draw_reaction(smarts)))

In [ ]:
prs.filter(
    pl.col("id") == "9c2c777c0fea421a26efc26857522fd2ccce1555"
)

In [ ]:
min_maps = pl.read_parquet(
    "/home/stef/bottle/artifacts/rxn_x_rule_mapping/mapped_known_reactions_x_rc_plus_0_rules.parquet"
)
min_maps.filter(
    pl.col("rxn_id") == "d7eea5a1a938257639a2cbe5950fa3501a3fc821"
)

In [ ]:
row['rxn_id']


In [ ]:
for row in min_maps.filter(pl.col("rule_id") == 515).iter_rows(named=True):
    smarts = row['smarts']
    print(row['rxn_id'])
    print(krs.filter(pl.col("id") == row['rxn_id'])['db_ids'])
    display(SVG(draw_reaction(smarts)))